## Assignment 4.1: Building a Plant Classification Model Using Convolutional Neural Networks
### Mostafa Zamaniturk

## Objective
Your task for this assignment is to build a plant classification model using CNNs and the Flavia dataset.

## Dataset
You will be using the Flavia datasetDownload Flavia dataset, which contains 1907 images of leaves from 32 different plant species. Each image is labeled with its corresponding species name.

### Note: 
This zipped file is 931MB and will take some time to download depending on your connection speed.

## Instructions
Preprocess the data: You will need to preprocess the Flavia dataset by resizing the images to a fixed size, converting them to grayscale, and splitting them into training, validation, and test sets.

Build a CNN model: You will need to design and implement a CNN model architecture that can effectively classify plant leaves based on their images. 

Train the model: You will need to train the CNN model on the preprocessed Flavia dataset using appropriate hyperparameters and regularization techniques.

Evaluate the model: You will need to evaluate the performance of the trained CNN model on the test set of the Flavia dataset using appropriate evaluation metrics such as accuracy, precision, recall, and F1 score.

Analyze the results: You will need to analyze the performance of the model and identify any potential areas for improvement. You can visualize the learned features of the model, plot confusion matrices, and perform other analysis techniques to gain insights into the model's behavior.

## Deliverables
Convert your Jupyter Notebook or Python script to a single PDF file or MS Word document. Your deliverable should contain your implementations of the tasks above, as well as any additional comments or observations you may have. Be sure to label each section of the notebook or script clearly with the corresponding task number. Please ensure the PDF or MS Word document displays the code and output appropriately.

## Environment Setup & Data Ingestion

In [0]:
# Install required packages for data ingestion and preprocessing
# TensorFlow will be installed later when needed for model training
print("Installing required packages...")
print("This may take a few moments...\n")

%pip install -q gdown Pillow scikit-learn matplotlib seaborn

print("\n✓ Packages installed successfully!")
print("\nReady to run the next cell for data ingestion.")

In [0]:
# Restart the Python environment to ensure new packages are loaded
dbutils.library.restartPython()

In [0]:
# Environment Setup & Data Ingestion for Flavia Plant Classification
import os
import zipfile
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import warnings
import shutil
warnings.filterwarnings('ignore')

# Scikit-learn utilities (TensorFlow will be imported later for model training)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("="*80)
print("FLAVIA PLANT CLASSIFICATION - ENVIRONMENT SETUP")
print("="*80)

print(f"\nPIL/Pillow available: Yes")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn available: Yes")

# Set random seeds for reproducibility
np.random.seed(42)

# Define paths
BASE_DIR = "/Workspace/Users/mzamaniturk@sandiego.edu/aai_511_assingments/4 (1)"
ZIP_FILE = os.path.join(BASE_DIR, "M4-archive.zip")
EXTRACT_DIR = os.path.join(BASE_DIR, "Flavia_dataset")

print(f"\nBase Directory: {BASE_DIR}")
print(f"ZIP File: {ZIP_FILE}")
print(f"Extract Directory: {EXTRACT_DIR}")

# Check if ZIP file exists
print("\n" + "="*80)
print("CHECKING DATASET AVAILABILITY")
print("="*80)

if not os.path.exists(ZIP_FILE):
    print(f"\n⚠ ZIP file not found at: {ZIP_FILE}")
    print("\n" + "="*80)
    print("MANUAL DOWNLOAD REQUIRED")
    print("="*80)
    print("\nGoogle Drive folder links cannot be downloaded programmatically.")
    print("\nPlease follow these steps:")
    print("\n1. Open this link in your browser:")
    print("   https://drive.google.com/drive/folders/1dvDZ2SAS60t6rM7saV8PmHDBVv-sbvu5")
    print("\n2. Download the file 'M4-archive.zip' to your computer")
    print("\n3. Upload it to Databricks at this location:")
    print(f"   {ZIP_FILE}")
    print("\n4. Once uploaded, re-run this cell")
    print("\n" + "="*80)
    
    # Stop here with a clear message instead of raising an error
    print("\n⏸ Execution paused - waiting for manual file upload")
    print("\nAfter uploading the file, run this cell again to continue.")
    
else:
    zip_size_mb = os.path.getsize(ZIP_FILE) / (1024 * 1024)
    print(f"\n✓ ZIP file found!")
    print(f"  Size: {zip_size_mb:.2f} MB")
    
    # Unzip the dataset
    print("\n" + "="*80)
    print("EXTRACTING DATASET")
    print("="*80)
    
    if os.path.exists(EXTRACT_DIR):
        print(f"\n✓ Dataset already extracted at: {EXTRACT_DIR}")
    else:
        print(f"\nExtracting {ZIP_FILE}...")
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_DIR)
        
        print(f"✓ Extraction complete!")
        print(f"  Extracted to: {EXTRACT_DIR}")
    
    # Explore dataset structure
    print("\n" + "="*80)
    print("DATASET STRUCTURE ANALYSIS")
    print("="*80)
    
    # Find the actual data directory (might be nested)
    data_dir = EXTRACT_DIR
    for root, dirs, files in os.walk(EXTRACT_DIR):
        # Look for directories with image files
        image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        if len(image_files) > 10:  # Found the main directory
            data_dir = root
            break
    
    print(f"\nData Directory: {data_dir}")
    
    # Parse all image files and extract species labels
    print("\n" + "="*80)
    print("PARSING IMAGE FILES & EXTRACTING LABELS")
    print("="*80)
    
    image_paths = []
    labels = []
    
    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                image_path = os.path.join(root, file)
                
                # Extract species label from filename
                # Flavia format: typically 1001.jpg to 3260.jpg
                # First 2 digits represent the species class (01-32)
                filename = os.path.splitext(file)[0]
                
                # Try to extract class ID from filename
                try:
                    # Flavia naming: first 2 digits are class ID
                    if len(filename) >= 2:
                        class_id = filename[:2]
                        image_paths.append(image_path)
                        labels.append(class_id)
                except:
                    continue
    
    print(f"\n✓ Found {len(image_paths)} images")
    
    # Count images per class
    label_counts = Counter(labels)
    num_classes = len(label_counts)
    
    print(f"\n✓ Number of classes: {num_classes}")
    print(f"\nImages per class (first 10):")
    for i, label in enumerate(sorted(label_counts.keys())[:10]):
        count = label_counts[label]
        print(f"  Class {label}: {count} images")
    if num_classes > 10:
        print(f"  ... and {num_classes - 10} more classes")
    
    # Verify expected dataset size
    print("\n" + "="*80)
    print("DATASET INTEGRITY CHECK")
    print("="*80)
    
    expected_classes = 32
    expected_images_approx = 1907
    
    print(f"\nExpected classes: {expected_classes}")
    print(f"Actual classes: {num_classes}")
    if num_classes == expected_classes:
        print("✓ Class count matches expected value!")
    else:
        print(f"⚠ Warning: Expected {expected_classes} classes, found {num_classes}")
    
    print(f"\nExpected images: ~{expected_images_approx}")
    print(f"Actual images: {len(image_paths)}")
    
    if abs(len(image_paths) - expected_images_approx) < 50:
        print("✓ Image count is within expected range!")
    else:
        print(f"⚠ Warning: Image count differs significantly from expected")
    
    # Analyze image properties
    print("\n" + "="*80)
    print("ANALYZING IMAGE PROPERTIES")
    print("="*80)
    
    print("\nSampling 10 random images to check dimensions and properties...")
    
    sample_indices = np.random.choice(len(image_paths), min(10, len(image_paths)), replace=False)
    image_dimensions = []
    image_channels = []
    
    for idx in sample_indices:
        img_path = image_paths[idx]
        try:
            img = Image.open(img_path)
            img_array = np.array(img)
            if img_array is not None:
                image_dimensions.append(img_array.shape[:2])  # (height, width)
                image_channels.append(img_array.shape[2] if len(img_array.shape) == 3 else 1)
        except:
            continue
    
    if image_dimensions:
        heights, widths = zip(*image_dimensions)
        print(f"\n✓ Sample image analysis:")
        print(f"  Height range: {min(heights)} - {max(heights)} pixels")
        print(f"  Width range: {min(widths)} - {max(widths)} pixels")
        print(f"  Channels: {Counter(image_channels)}")
        print(f"  Most common dimension: {Counter(image_dimensions).most_common(1)[0][0]}")
    
    # Plot sample images with labels
    print("\n" + "="*80)
    print("VISUALIZING SAMPLE IMAGES")
    print("="*80)
    
    print("\nPlotting 16 random sample images...")
    
    # Select 16 random samples from different classes if possible
    num_samples = 16
    sample_per_class = {}
    
    for img_path, label in zip(image_paths, labels):
        if label not in sample_per_class:
            sample_per_class[label] = []
        sample_per_class[label].append(img_path)
    
    # Try to get diverse samples
    sample_paths = []
    sample_labels = []
    
    for label in sorted(sample_per_class.keys())[:num_samples]:
        if sample_per_class[label]:
            sample_paths.append(sample_per_class[label][0])
            sample_labels.append(label)
    
    # Fill remaining with random samples
    while len(sample_paths) < num_samples and len(sample_paths) < len(image_paths):
        idx = np.random.randint(0, len(image_paths))
        if image_paths[idx] not in sample_paths:
            sample_paths.append(image_paths[idx])
            sample_labels.append(labels[idx])
    
    # Create visualization
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    axes = axes.flatten()
    
    for idx, (img_path, label) in enumerate(zip(sample_paths, sample_labels)):
        if idx >= 16:
            break
        
        # Read image using PIL
        try:
            img = Image.open(img_path)
            img_array = np.array(img)
            
            # Display
            axes[idx].imshow(img)
            axes[idx].set_title(f'Class {label}\nShape: {img_array.shape[:2]}', 
                               fontsize=10, fontweight='bold')
            axes[idx].axis('off')
        except Exception as e:
            axes[idx].text(0.5, 0.5, 'Failed to load', 
                          ha='center', va='center')
            axes[idx].axis('off')
    
    plt.suptitle('Flavia Dataset - Sample Images (Raw, Unprocessed)', 
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Sample images displayed!")
    
    # Summary statistics
    print("\n" + "="*80)
    print("DATASET SUMMARY")
    print("="*80)
    
    print(f"\n📊 Dataset Statistics:")
    print(f"  Total images: {len(image_paths)}")
    print(f"  Number of classes (species): {num_classes}")
    print(f"  Images per class: {len(image_paths) / num_classes:.1f} (average)")
    print(f"  Min images per class: {min(label_counts.values())}")
    print(f"  Max images per class: {max(label_counts.values())}")
    
    print(f"\n🔍 Observations:")
    print(f"  • Images have varying dimensions (will need resizing)")
    print(f"  • Images are in color (RGB) - will convert to grayscale as per requirements")
    print(f"  • Dataset appears balanced across classes")
    print(f"  • Lighting and orientation vary across samples")
    
    print("\n" + "="*80)
    print("ENVIRONMENT SETUP COMPLETE!")
    print("="*80)
    print("\n✓ Libraries imported")
    print("✓ Dataset extracted and verified")
    print("✓ {0} images from {1} plant species identified".format(len(image_paths), num_classes))
    print("\n➡️  Ready for preprocessing (Cell 11)")
    print("="*80)

1. Preprocess the data: 

You will need to preprocess the Flavia dataset by resizing the images to a fixed size, converting them to grayscale, and splitting them into training, validation, and test sets.